#imports and setup

In [2]:
import requests
import time
import pandas as pd

BASE = "https://collectionapi.metmuseum.org/public/collection/v1"

# working taxonomy 
CULTURES = [
    "Greek", "Roman", "Egyptian", "Etruscan", "Chinese",
    "Mesoamerican", "Native American", "Persian",
    "Minoan", "Nok", "Modern"
]

PERIOD_BUCKETS = [
    "3000-1200BCE", "1200-800BCE", "800-300BCE",
    "300BCE-300CE", "300-1000CE", "1000-1500CE", "1500CE-present"
]

#api fetch functions

In [3]:
def search_objects(query, has_images=True):
    r = requests.get(f"{BASE}/search", params={
        "q": query,
        "hasImages": "true" if has_images else "false"
    })
    r.raise_for_status()
    return r.json().get("objectIDs", []) or []


def get_object(object_id):
    r = requests.get(f"{BASE}/objects/{object_id}")
    r.raise_for_status()
    return r.json()


#vessel filter

In [ ]:
# vessel object names we'll accept (lowercase, partial match)
VESSEL_KEYWORDS = [
    "vase", "vessel", "amphora", "jar", "jug", "pot", "bowl",
    "krater", "kylix", "oinochoe", "hydria", "lekythos", "urn",
    "pitcher", "flask", "vial", "cup", "dish", "ewer", "olpe",
    "pyxis", "situla", "askos"
]

def is_vessel(object_name):
    if not object_name:
        return False
    name = object_name.lower()
    return any(kw in name for kw in VESSEL_KEYWORDS)



,id,has_image,culture,date_begin,date_end,object_name,is_vessel
0,254896,True,"Greek, Attic",-490,-490,Amphora,True
1,436105,True,,1787,1787,Painting,False
2,248698,True,"Greek, Attic",-490,-490,Kylix,True
3,248500,True,"Greek, Attic",-530,-530,Grave stele of a youth and a little girl,False
4,459028,True,,1480,1500,Painting,False


#collect vessels by department

In [ ]:
import random
import time

def get_department_ids(dept_id, max_retries=3):
    for attempt in range(max_retries):
        try:
            r = requests.get(f"{BASE}/objects", params={"departmentIds": dept_id})
            if r.status_code == 403:
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            return set(r.json().get("objectIDs", []) or [])
        except requests.exceptions.RequestException:
            if attempt == max_retries - 1:
                return set()
            time.sleep(2 ** attempt)
    return set()

def sample_department_vessels(dept_id, sample_size=400):
    dept_ids = get_department_ids(dept_id)
    sample_ids = random.sample(list(dept_ids), min(sample_size, len(dept_ids)))
    
    records = []
    for i, oid in enumerate(sample_ids):
        obj = get_object_safe(oid)
        if obj and is_vessel(obj.get("objectName", "")):
            records.append({
                "id": oid,
                "has_image": bool(obj.get("primaryImage")),
                "culture": obj.get("culture"),
                "date_begin": obj.get("objectBeginDate"),
                "date_end": obj.get("objectEndDate"),
                "object_name": obj.get("objectName"),
                "department": dept_id
            })
        time.sleep(0.3)
        if i % 100 == 0:
            print(f"  dept {dept_id} progress: {i}/{len(sample_ids)}")
    return records

all_vessel_records = []
for dept_id in DEPTS.keys():
    print(f"Sampling department {dept_id} ({DEPTS[dept_id]})...")
    recs = sample_department_vessels(dept_id, sample_size=400)
    print(f"  -> {len(recs)} vessels found")
    all_vessel_records.extend(recs)

full_df = pd.DataFrame(all_vessel_records)
print(f"\nTotal vessels across all departments: {len(full_df)}")
print(full_df["culture"].value_counts())

full_df.to_csv("full_df_backup.csv", index=False)
print("\nSaved to full_df_backup.csv")

Sampling department 13 (Greek and Roman Art)...
  dept 13 progress: 0/400
  dept 13 progress: 100/400
  dept 13 progress: 200/400
  dept 13 progress: 300/400
  -> 223 vessels found
Sampling department 10 (Egyptian Art)...
  dept 10 progress: 0/400
  dept 10 progress: 100/400
  dept 10 progress: 200/400
  dept 10 progress: 300/400
  -> 60 vessels found
Sampling department 3 (Ancient West Asian Art)...
  dept 3 progress: 0/400
  dept 3 progress: 100/400
  dept 3 progress: 200/400
  dept 3 progress: 300/400
  -> 47 vessels found
Sampling department 5 (Arts of Africa, Oceania, and the Americas)...
  dept 5 progress: 0/400
  dept 5 progress: 100/400
  dept 5 progress: 200/400
  dept 5 progress: 300/400
  -> 30 vessels found
Sampling department 6 (Asian Art)...
  dept 6 progress: 0/400
  dept 6 progress: 100/400
  dept 6 progress: 200/400
  dept 6 progress: 300/400
  -> 42 vessels found

Total vessels across all departments: 402
culture
Greek, Attic            164
                         73

#build culture map 

In [ ]:
culture_map = {
    # Greek (broad bucket - includes regional styles + Cypriot)
    "Greek, Attic": "Greek",
    "Greek": "Greek",
    "Greek, Boeotian": "Greek",
    "Greek, Laconian": "Greek",
    "Greek, South Italian, Apulian": "Greek",
    "Greek, South Italian, Campanian": "Greek",
    "Greek, South Italian, Paestan": "Greek",
    "Greek, South Italian, Apulian, Canosan": "Greek",
    "East Greek/Sardis, Lydian": "Greek",
    "Cypriot": "Greek",

    # Aegean Bronze Age (Minoan/Mycenaean family)
    "Minoan": "Aegean",
    "Mycenaean": "Aegean",
    "Aegean": "Aegean",
    "Helladic": "Aegean",
    "Cycladic": "Aegean",

    # Roman
    "Roman": "Roman",
    "Roman, Gaul": "Roman",
    "Roman, Cypriot": "Roman",

    # Etruscan / Italic
    "Etruscan": "Etruscan",
    "Italic-Native, Sicilian (Centuripe)": "Etruscan",

    # Egyptian
    "Egyptian": "Egyptian",

    # Near East / Persian (broad bucket)
    "Iran": "Near Eastern",
    "Iranian": "Near Eastern",
    "Sasanian": "Near Eastern",
    "Parthian": "Near Eastern",
    "Achaemenid": "Near Eastern",
    "Sumerian": "Near Eastern",
    "Israelite": "Near Eastern",
    "Assyrian": "Near Eastern",
    "Hittite": "Near Eastern",

    # East Asian (broad bucket)
    "China": "East Asian",
    "Chinese": "East Asian",
    "Japan": "East Asian",
    "Korea": "East Asian",

    # Andean (broad bucket)
    "Paracas": "Andean",
    "Inca": "Andean",
    "Chimú": "Andean",
    "Nasca": "Andean",
    "Manteño Huancavilca": "Andean",
    "Tairona": "Andean",

    # Mesoamerican
    "Maya": "Mesoamerican",
    "Aztec": "Mesoamerican",

    # Sub-Saharan African (broad bucket, excludes Egypt)
    "Nok": "Sub-Saharan African",
    "Igbo-Ukwu": "Sub-Saharan African",
    "Yoruba": "Sub-Saharan African",
    "Benin": "Sub-Saharan African",
    "Ashanti": "Sub-Saharan African",
    "Akan": "Sub-Saharan African",
    "Luba peoples": "Sub-Saharan African",
    "Great Zimbabwe": "Sub-Saharan African",
    "Mapungubwe": "Sub-Saharan African",
    "Dogon": "Sub-Saharan African",
}

full_df["culture_clean"] = full_df["culture"].map(culture_map).fillna("Other/Unmapped")
full_df["culture_specific"] = full_df["culture"]  # keep original label for reference

print(full_df["culture_clean"].value_counts())

culture_clean
Greek             184
Other/Unmapped    127
East Asian         41
Near Eastern       21
Roman              10
Andean              8
Etruscan            6
Aegean              5
Name: count, dtype: int64


In [ ]:
additional_mappings = {
    "Baule peoples": "Sub-Saharan African",
    "Bamana peoples": "Sub-Saharan African",
    "Bamum (Bamendjing)": "Sub-Saharan African",
    "Bamenda": "Sub-Saharan African",
    "Zulu peoples": "Sub-Saharan African",
    "Asante": "Sub-Saharan African",
    "Mexica (Aztec)": "Mesoamerican",
    "Veracruz": "Mesoamerican",
    "Mexican": "Mesoamerican",
    "Olmec Tradition artist, Central Highlands, Mexico": "Mesoamerican",
    "Quimbaya": "Andean",
    "Chorrera": "Andean",
    "Inca": "Andean",
    "Mogollon (Mimbres)": "Native American",
    "Pueblo": "Native American",
}

culture_map.update(additional_mappings)

full_df["culture_clean"] = full_df["culture"].map(culture_map).fillna("Other/Unmapped")
full_df.loc[full_df["department"] == 10, "culture_clean"] = "Egyptian"  # fix: Egyptian dept has blank culture field
full_df["culture_specific"] = full_df["culture"]

print(full_df["culture_clean"].value_counts())

# save so this never has to be redone
full_df.to_csv("full_df_backup.csv", index=False)
print("\nSaved.")

culture_clean
Greek              184
Other/Unmapped      64
Egyptian            60
East Asian          41
Near Eastern        21
Roman               10
Andean               8
Etruscan             6
Aegean               5
Mesoamerican         2
Native American      1
Name: count, dtype: int64

Saved.


#filter to solid classes

In [ ]:
KEEP_CLASSES = ["Greek", "Egyptian", "East Asian", "Near Eastern", "Roman"]

train_df = full_df[
    (full_df["culture_clean"].isin(KEEP_CLASSES)) &
    (full_df["has_image"] == True)
].reset_index(drop=True)

print(train_df["culture_clean"].value_counts())
print(f"Total: {len(train_df)}")

culture_clean
Greek           177
East Asian       39
Near Eastern     21
Egyptian         17
Roman             8
Name: count, dtype: int64
Total: 262


#fetch image urls

In [ ]:
image_urls = []
for oid in train_df["id"]:
    obj = get_object_safe(oid)
    url = obj.get("primaryImage") if obj else None
    image_urls.append(url)
    time.sleep(0.2)

train_df["image_url"] = image_urls
print(train_df["image_url"].isna().sum(), "missing URLs out of", len(train_df))

6 missing URLs out of 262


#train val split

In [ ]:
from sklearn.model_selection import train_test_split

counts = train_df["culture_clean"].value_counts()
print("Class counts before split:")
print(counts)

if counts.min() < 2:
    print("\nAt least one class has fewer than 2 samples; using a non-stratified split.")
    train_split, val_split = train_test_split(
        train_df, test_size=0.2, random_state=42
    )
else:
    train_split, val_split = train_test_split(
        train_df,
        test_size=0.2,
        stratify=train_df["culture_clean"],
        random_state=42,
    )

print("\nTrain:")
print(train_split["culture_clean"].value_counts())
print("\nVal:")
print(val_split["culture_clean"].value_counts())


NameError: name 'train_df' is not defined

#downlad images

In [ ]:
import os

os.makedirs("data/images", exist_ok=True)

def download_images(df, label=""):
    failed = []
    for i, row in df.iterrows():
        fname = f"data/images/{row['id']}.jpg"
        if os.path.exists(fname):
            continue
        try:
            r = requests.get(row["image_url"], timeout=10)
            r.raise_for_status()
            with open(fname, "wb") as f:
                f.write(r.content)
        except Exception:
            failed.append(row["id"])
        time.sleep(0.2)
    print(f"{label}: {len(df) - len(failed)}/{len(df)} downloaded successfully")
    return failed

train_failed = download_images(train_split, "Train")
val_failed = download_images(val_split, "Validation")

Train: 202/209 downloaded successfully
Validation: 53/53 downloaded successfully


#dataset + transformations

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
import torch.nn as nn
from PIL import Image

CULTURE_CLASSES = sorted(train_split["culture_clean"].unique())
culture2idx = {c: i for i, c in enumerate(CULTURE_CLASSES)}
idx2culture = {i: c for c, i in culture2idx.items()}
print(culture2idx)

class PotteryDataset(Dataset):
    def __init__(self, df, culture2idx, transform=None):
        self.df = df.reset_index(drop=True)
        self.culture2idx = culture2idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f"data/images/{row['id']}.jpg").convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.culture2idx[row["culture_clean"]]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = PotteryDataset(train_split, culture2idx, transform=train_transform)
val_dataset = PotteryDataset(val_split, culture2idx, transform=val_transform)
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

# sanity check: load one sample from the training set
img, label = train_dataset[0]
print(f"Sample image shape: {img.shape}, label: {label} ({idx2culture[label]})")

NameError: name 'train_split' is not defined

#model definition

In [ ]:
import os

torch.set_num_threads(max(1, os.cpu_count() // 2))

class PotteryNet(nn.Module):
    def __init__(self, n_cultures):
        super().__init__()
        backbone = models.resnet18(weights=None)
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        for p in self.features.parameters():
            p.requires_grad = False
        self.culture_head = nn.Linear(512, n_cultures)

    def forward(self, x):
        x = self.features(x).flatten(1)
        return self.culture_head(x)

device = torch.device("cpu")
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", getattr(torch.backends, "mps", None).is_available() if hasattr(torch.backends, "mps") else False)
print(f"Using device: {device}")
model = PotteryNet(n_cultures=len(culture2idx)).to(device)

NameError: name 'torch' is not defined

#dataloaders oversampling

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler

# oversample based on class frequency in the training set
class_counts = train_split["culture_clean"].value_counts()
sample_weights = train_split["culture_clean"].map(lambda c: 1.0 / class_counts[c]).values

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=8, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)

# sanity check
imgs, labels = next(iter(train_loader))
print(f"Batch shape: {imgs.shape}, labels: {labels}")

Batch shape: torch.Size([16, 3, 224, 224]), labels: tensor([4, 3, 4, 0, 4, 4, 3, 1, 1, 0, 0, 1, 2, 0, 4, 1])


#training loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

n_epochs = 15
patience = 3
best_val_acc = 0
epochs_no_improve = 0

for epoch in range(n_epochs):
    model.train()
    train_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    print(f"Epoch {epoch+1}/{n_epochs} - train_loss: {train_loss/len(train_loader):.4f} - val_acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), "best_pottery_model_cpu.pt")
        print(f"  -> new best model saved (val_acc: {val_acc:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"\nEarly stopping triggered after {epoch+1} epochs")
            print(f"Best val_acc: {best_val_acc:.4f}")
            break

model.load_state_dict(torch.load("best_pottery_model_cpu.pt"))
print("Loaded best model checkpoint.")

Epoch 1/50 - train_loss: 0.0137 - val_acc: 0.7647
  -> new best model saved (val_acc: 0.7647)
Epoch 2/50 - train_loss: 0.0103 - val_acc: 0.7647
Epoch 3/50 - train_loss: 0.0177 - val_acc: 0.7647
Epoch 4/50 - train_loss: 0.0113 - val_acc: 0.8039
  -> new best model saved (val_acc: 0.8039)
Epoch 5/50 - train_loss: 0.0096 - val_acc: 0.7843
Epoch 6/50 - train_loss: 0.0042 - val_acc: 0.8039
Epoch 7/50 - train_loss: 0.0074 - val_acc: 0.7843
Epoch 8/50 - train_loss: 0.0053 - val_acc: 0.8039
Epoch 9/50 - train_loss: 0.0209 - val_acc: 0.7647
Epoch 10/50 - train_loss: 0.0081 - val_acc: 0.8039
Epoch 11/50 - train_loss: 0.0067 - val_acc: 0.7647

Early stopping triggered after 11 epochs
Best val_acc: 0.8039
Loaded best model checkpoint.


#model eval

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=[idx2culture[i] for i in range(len(idx2culture))]))

              precision    recall  f1-score   support

  East Asian       0.69      1.00      0.82         9
    Egyptian       0.25      0.20      0.22         5
       Greek       0.91      0.97      0.94        31
Near Eastern       1.00      0.25      0.40         4
       Roman       0.00      0.00      0.00         2

    accuracy                           0.80        51
   macro avg       0.57      0.48      0.48        51
weighted avg       0.78      0.80      0.77        51



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
